In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import matplotlib.pylab as plt
import seaborn as sns

from dmba import regressionSummary, exhaustive_search
from dmba import backward_elimination, forward_selection, stepwise_selection
from dmba import adjusted_r2_score, AIC_score, BIC_score

In [3]:
# Big dataset
# Read CSV and create data frame from games where GSW are at home
filename = "Detailed NBA Data (04-20)/games.csv"
df = pd.read_csv(filename)
GSW_filter = df['TEAM_ID_home'] == 1610612744
df = df[GSW_filter]

# Rename columns for GSW's stats
df['Team'] = 'GSW'
df['Date'] = df['GAME_DATE_EST']
df['Points'] = df['PTS_home']
df['FG_PCT'] = df['FG_PCT_home']
df['FT_PCT'] = df['FT_PCT_home']
df['FG3_PCT'] = df['FG3_PCT_home']
df['AST'] = df['AST_home']
df['REB'] = df['REB_home']

# Create opponent stats
df['Opp.Points'] = df['PTS_away']
df['Opp.FG_PCT'] = df['FG_PCT_away']
df['Opp.FT_PCT'] = df['FT_PCT_away']
df['Opp.FG3_PCT'] = df['FG3_PCT_away']
df['Opp.AST'] = df['AST_away']
df['Opp.REB'] = df['REB_away']

# Numerical indicator of who wins
df['ScoreDiff'] = df['Points'] - df['Opp.Points']

# Delete columns not needed
df = df.drop(['HOME_TEAM_WINS','GAME_ID', 'GAME_STATUS_TEXT', 'TEAM_ID_home', 'TEAM_ID_away', 
                        'HOME_TEAM_ID', 'VISITOR_TEAM_ID', 'GAME_DATE_EST', 'PTS_home', 'FG_PCT_home', 
                        'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'PTS_away', 'FG_PCT_away', 
                        'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away'], axis = 1)

# Set 'Date' to a date format and sort
df['Date'] = pd.to_datetime(df.Date)
df = df.sort_values(by = 'Date')
df.reset_index(inplace = True)
df = df.drop(['index'], axis = 1)

df.to_csv(r'C:\Users\Darion\Desktop\NBA Project\Large_NBA.csv', 
              index = False, header=True)

In [4]:
# Obtain dataset of games from 10/29/2014 to 4/10/20 (14-15 to 17-18 regular season)
filename = "NBA Games Data (14-18)/nba.games.stats.csv"
data = pd.read_csv(filename)

# Create filter for GSW games
GSW_filter = data['Team'] == 'GSW'
data = data[GSW_filter]

# Create point differential
data['ScoreDiff'] = data['TeamPoints'] - data['OpponentPoints']
data['Opp.ScoreDiff'] = data['OpponentPoints'] - data['TeamPoints']

# Drop columns we don't need
data = data.drop(['Unnamed: 0', 'Game', 'Home', 'Opponent', 'WINorLOSS', 'TeamPoints', 
                      'OpponentPoints', 'OffRebounds', 'TotalFouls','Opp.OffRebounds',
                      'Opp.TotalFouls'], axis = 1) 

# Change date format, reset index, preview with head()
data['Date'] = pd.to_datetime(data.Date)
data = data.sort_values(by = 'Date')
data.reset_index(inplace = True)
data = data.drop(['index'], axis = 1)
data.head()

,Team,Date,FieldGoals,FieldGoalsAttempted,FieldGoals.,X3PointShots,X3PointShotsAttempted,X3PointShots.,FreeThrows,FreeThrowsAttempted,...,Opp.FreeThrows,Opp.FreeThrowsAttempted,Opp.FreeThrows.,Opp.TotalRebounds,Opp.Assists,Opp.Steals,Opp.Blocks,Opp.Turnovers,ScoreDiff,Opp.ScoreDiff
0,GSW,2014-10-29,33,75,0.440,6,27,0.222,23,32,...,26,35,0.743,50,13,11,3,26,18,-18
1,GSW,2014-11-01,46,83,0.554,11,23,0.478,24,28,...,18,25,0.720,37,21,9,3,22,23,-23
2,GSW,2014-11-02,36,81,0.444,6,19,0.316,17,21,...,9,14,0.643,56,20,9,2,19,5,-5
3,GSW,2014-11-05,43,74,0.581,15,25,0.600,20,20,...,21,27,0.778,30,26,12,2,14,17,-17
4,GSW,2014-11-08,39,87,0.448,9,28,0.321,11,17,...,17,21,0.810,46,13,14,0,22,11,-11


In [5]:
# Export as CSV
data.to_csv(r'C:\Users\Darion\Desktop\NBA Project\GSW.csv', 
              index = False, header=True)

In [6]:
# Multiple regression
Data_X = data.drop(['Team', 'Date', 'ScoreDiff', 'Opp.ScoreDiff'], axis = 1)
                #'Opp.FieldGoals',
               #'Opp.FieldGoalsAttempted', 'Opp.FieldGoals.', 'Opp.3PointShots',
               #'Opp.3PointShotsAttempted', 'Opp.3PointShots.', 'Opp.FreeThrows',
               #'Opp.FreeThrowsAttempted', 'Opp.FreeThrows.', 'Opp.TotalRebounds',
               #'Opp.Assists', 'Opp.Steals', 'Opp.Blocks', 'Opp.Turnovers',], axis = 1)
Data_y = data['ScoreDiff'].values.reshape(-1, 1)


In [7]:
Data_X2 = sm.add_constant(Data_X)
model = sm.OLS(Data_y, Data_X2)
model.fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 4.735e+28
Date:                Tue, 04 May 2021   Prob (F-statistic):               0.00
Time:                        00:41:14   Log-Likelihood:                 9110.5
No. Observations:                 328   AIC:                        -1.816e+04
Df Residuals:                     299   BIC:                        -1.805e+04
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                    -1.194e-12   2.07e-12     -0.577      0.564   -5.27e-12    2.88e-12
FieldGoals                   2.0000   3.58e-14   5.58e+13      0.000       2.000       2.000
FieldGoalsAttempted       3.331e-16   1.76e-14      0.019      0.985   -3.42e-14    3.49e-14
FieldGoals.              -2.274e-13   3.13e-12     -0.073      0.942   -6.39e-12    5.93e-12
X3PointShots                 1.0000   2.56e-14    3.9e+13      0.000       1.000       1.000
X3PointShotsAttempted      6.55e-15   1.05e-14      0.622      0.535   -1.42e-14    2.73e-14
X3PointShots.             4.086e-13   7.65e-13      0.534      0.594    -1.1e-12    1.91e-12
FreeThrows                   1.0000    1.8e-14   5.55e+13      0.000       1.000       1.000
FreeThrowsAttempted      -4.885e-15   1.44e-14     -0.339      0.735   -3.32e-14    2.34e-14
FreeThrows.              -9.504e-14   3.57e-13     -0.266      0.791   -7.99e-13    6.08e-13
TotalRebounds            -1.006e-15   4.65e-15     -0.216      0.829   -1.02e-14    8.15e-15
Assists                   5.412e-16   3.64e-15      0.149      0.882   -6.61e-15     7.7e-15
Steals                    3.549e-15   6.97e-15      0.509      0.611   -1.02e-14    1.73e-14
Blocks                    1.388e-16   4.85e-15      0.029      0.977   -9.41e-15    9.69e-15
Turnovers                -2.637e-15   7.36e-15     -0.358      0.720   -1.71e-14    1.19e-14
Opp.FieldGoals              -2.0000   3.96e-14  -5.05e+13      0.000      -2.000      -2.000
Opp.FieldGoalsAttempted   1.554e-14   1.78e-14      0.871      0.385   -1.96e-14    5.07e-14
Opp.FieldGoals.           2.615e-12   3.47e-12      0.754      0.451   -4.21e-12    9.44e-12
Opp.3PointShots             -1.0000   2.04e-14   -4.9e+13      0.000      -1.000      -1.000
Opp.3PointShotsAttempted  3.664e-15   7.14e-15      0.513      0.608   -1.04e-14    1.77e-14
Opp.3PointShots.          2.274e-13   5.21e-13      0.437      0.663   -7.97e-13    1.25e-12
Opp.FreeThrows              -1.0000   1.94e-14  -5.16e+13      0.000      -1.000      -1.000
Opp.FreeThrowsAttempted  -4.996e-16   1.48e-14     -0.034      0.973   -2.96e-14    2.86e-14
Opp.FreeThrows.          -2.132e-14    4.5e-13     -0.047      0.962   -9.06e-13    8.64e-13
Opp.TotalRebounds         1.249e-15   5.06e-15      0.247      0.805   -8.72e-15    1.12e-14
Opp.Assists              -3.816e-16   3.58e-15     -0.107      0.915   -7.43e-15    6.67e-15
Opp.Steals                9.437e-16   6.22e-15      0.152      0.880   -1.13e-14    1.32e-14
Opp.Blocks                 3.99e-16   6.36e-15      0.063      0.950   -1.21e-14    1.29e-14
Opp.Turnovers            -6.883e-15   7.46e-15     -0.922      0.357   -2.16e-14     7.8e-15
==============================================================================
Omnibus:                    

In [8]:
# Adjust independent variables
Data_X = data.drop(['Team', 'Date', 'ScoreDiff', 'Opp.ScoreDiff', 'FieldGoalsAttempted',
                   'FieldGoals.', 'X3PointShotsAttempted', 'X3PointShots.',
                   'FreeThrowsAttempted', 'FreeThrows.', 'TotalRebounds', 'Assists',
                   'Steals', 'Blocks', 'Turnovers', 'Opp.FieldGoalsAttempted',
                   'Opp.FieldGoals.', 'Opp.3PointShotsAttempted', 'Opp.3PointShots.',
                   'Opp.FreeThrowsAttempted', 'Opp.FreeThrows.', 'Opp.TotalRebounds',
                   'Opp.Assists', 'Opp.Steals', 'Opp.Blocks', 'Opp.Turnovers'], axis = 1)
Data_X2 = sm.add_constant(Data_X)
model = sm.OLS(Data_y, Data_X2)
model.fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 2.057e+30
Date:                Tue, 04 May 2021   Prob (F-statistic):               0.00
Time:                        00:45:44   Log-Likelihood:                 9464.7
No. Observations:                 328   AIC:                        -1.892e+04
Df Residuals:                     321   BIC:                        -1.889e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            4.441e-16   4.63e-14      0.010      0.992   -9.06e-14    9.15e-14
FieldGoals          2.0000   9.46e-16   2.11e+15      0.000       2.000       2.000
X3PointShots        1.0000   1.22e-15   8.21e+14      0.000       1.000       1.000
FreeThrows          1.0000    7.4e-16   1.35e+15      0.000       1.000       1.000
Opp.FieldGoals     -2.0000   8.86e-16  -2.26e+15      0.000      -2.000      -2.000
Opp.3PointShots    -1.0000   1.22e-15   -8.2e+14      0.000      -1.000      -1.000
Opp.FreeThrows     -1.0000   7.23e-16  -1.38e+15      0.000      -1.000      -1.000
==============================================================================
Omnibus:                        8.697   Durbin-Watson:                   0.095
Prob(Omnibus):                  0.013   Jarque-Bera (JB):                8.603
Skew:                          -0.375   Prob(JB):                       0.0136
Kurtosis:                       3.256   Cond. No.                         751.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [10]:
Data_X = data.drop(['Team', 'Date', 'ScoreDiff', 'Opp.ScoreDiff', 'FieldGoals',
                    'X3PointShots', 'FreeThrows', 'Opp.FieldGoals', 'Opp.3PointShots',
                    'FieldGoalsAttempted', 'FieldGoals.', 'X3PointShotsAttempted', 'X3PointShots.',
                    'FreeThrowsAttempted', 'FreeThrows.', 'Opp.FieldGoalsAttempted',
                    'Opp.FieldGoals.', 'Opp.3PointShotsAttempted', 'Opp.3PointShots.',
                    'Opp.FreeThrowsAttempted', 'Opp.FreeThrows.'], axis = 1)
Data_X2 = sm.add_constant(Data_X)
model = sm.OLS(Data_y, Data_X2)
model.fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.620
Model:                            OLS   Adj. R-squared:                  0.607
Method:                 Least Squares   F-statistic:                     46.88
Date:                Tue, 04 May 2021   Prob (F-statistic):           7.60e-60
Time:                        00:49:08   Log-Likelihood:                -1170.9
No. Observations:                 328   AIC:                             2366.
Df Residuals:                     316   BIC:                             2411.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 2.6170      7.981      0.328      0.743     -13.085      18.319
TotalRebounds         0.6982      0.087      8.001      0.000       0.527       0.870
Assists               1.0848      0.102     10.616      0.000       0.884       1.286
Steals                0.2465      0.270      0.912      0.363      -0.286       0.778
Blocks                0.4522      0.178      2.539      0.012       0.102       0.803
Turnovers            -0.8461      0.203     -4.176      0.000      -1.245      -0.448
Opp.FreeThrows       -0.2424      0.085     -2.855      0.005      -0.410      -0.075
Opp.TotalRebounds    -0.6810      0.094     -7.240      0.000      -0.866      -0.496
Opp.Assists          -0.9124      0.111     -8.206      0.000      -1.131      -0.694
Opp.Steals           -0.3484      0.245     -1.422      0.156      -0.830       0.134
Opp.Blocks           -0.2294      0.233     -0.982      0.327      -0.689       0.230
Opp.Turnovers         0.6818      0.219      3.120      0.002       0.252       1.112
==============================================================================
Omnibus:                        2.487   Durbin-Watson:                   2.032
Prob(Omnibus):                  0.288   Jarque-Bera (JB):                2.510
Skew:                          -0.210   Prob(JB):                        0.285
Kurtosis:                       2.914   Cond. No.                     1.30e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.3e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""